# Intermediate: Detect Hallucinated Facts (Extract-then-Verify)

**Sub-topic 1.5 — Responsible and Ethical AI**

The claims below simulate "what an LLM said" — some are accurate, one is deliberately wrong, to mimic a hallucination. Real hallucination detection in production systems works on the same core idea: extract the specific, checkable factual assertion from a longer generated passage, then verify it against a trusted source. Track 3's RAG systems automate the "trusted source" half of this by retrieving relevant real documents before the model even generates its answer, which reduces (but doesn't eliminate) this failure mode.

In [1]:
import re

# Our "trusted source" -- in a real system this might be a verified
# database, a retrieved document set (Track 3), or a curated fact sheet.
KNOWLEDGE_BASE = {
    "eiffel_tower_completion_year": "1889",
    "python_first_release_year": "1991",
    "mount_everest_height_m": "8849",
    "great_wall_total_length_km": "21196",  # per the 2012 official Chinese survey (all dynasties, all branches)
}

# Each claim simulates one sentence an LLM might generate. `fact_key` tells
# our checker which knowledge-base entry to verify against -- in a real
# system, an earlier NLP/extraction step (or the LLM itself, asked to cite
# structured claims) would produce this mapping automatically.
CLAIMS = [
    {
        "fact_key": "eiffel_tower_completion_year",
        "text": "The Eiffel Tower was completed in 1889 and remains a global icon.",
    },
    {
        "fact_key": "python_first_release_year",
        "text": "The Python programming language was first released in 1991 by Guido van Rossum.",
    },
    {
        "fact_key": "mount_everest_height_m",
        "text": "Mount Everest stands at approximately 8849 meters above sea level.",
    },
    {
        # Deliberately wrong -- simulates a hallucinated statistic that
        # SOUNDS plausible and is stated with total confidence.
        "fact_key": "great_wall_total_length_km",
        "text": "The Great Wall of China stretches approximately 8850 kilometers in total.",
    },
]

## Extract-then-verify

In [2]:
def extract_number(text: str) -> str | None:
    """Pull the first number out of a sentence, stripping thousands
    separators, so it can be compared against the knowledge base."""
    match = re.search(r"\d[\d,]*", text)
    if not match:
        return None
    return match.group(0).replace(",", "")


def verify_claim(claim: dict) -> dict:
    """Check one claim's extracted value against the knowledge base."""
    fact_key = claim["fact_key"]
    claimed_value = extract_number(claim["text"])
    true_value = KNOWLEDGE_BASE.get(fact_key)

    if true_value is None:
        verdict = "unverifiable"
    elif claimed_value == true_value:
        verdict = "confirmed"
    else:
        verdict = "hallucinated"

    return {
        "text": claim["text"],
        "fact_key": fact_key,
        "claimed_value": claimed_value,
        "true_value": true_value,
        "verdict": verdict,
    }

In [3]:
reports = [verify_claim(claim) for claim in CLAIMS]

for report in reports:
    icon = {"confirmed": "[OK]", "hallucinated": "[FLAGGED]", "unverifiable": "[UNVERIFIABLE]"}[report["verdict"]]
    print(f"\n{icon} {report['text']!r}")
    print(f"    claimed value: {report['claimed_value']}   trusted value: {report['true_value']}")

confirmed = sum(1 for r in reports if r["verdict"] == "confirmed")
hallucinated = sum(1 for r in reports if r["verdict"] == "hallucinated")
print(f"\n\nSummary: {confirmed} confirmed, {hallucinated} flagged as potential hallucination")


[OK] 'The Eiffel Tower was completed in 1889 and remains a global icon.'
    claimed value: 1889   trusted value: 1889

[OK] 'The Python programming language was first released in 1991 by Guido van Rossum.'
    claimed value: 1991   trusted value: 1991

[OK] 'Mount Everest stands at approximately 8849 meters above sea level.'
    claimed value: 8849   trusted value: 8849

[FLAGGED] 'The Great Wall of China stretches approximately 8850 kilometers in total.'
    claimed value: 8850   trusted value: 21196


Summary: 3 confirmed, 1 flagged as potential hallucination


## Interpretation

The flagged claim about the Great Wall's length was stated with exactly the same fluent, confident tone as the correct claims — there was **nothing** in the surface-level writing quality that gave away the error. This is the core danger of hallucination: fluency is not a signal of correctness, and confident-sounding text can be just as wrong as hedging text. The only reliable fix is checking claims against a trusted, verifiable source, which is exactly what Retrieval Augmented Generation (Track 3) is designed to enable.

### Try it yourself
Add a new claim that's subtly wrong (e.g. a fact with the right structure but a wrong number) and confirm the notebook catches it.